# Retrain and save the selected model: CNN (words)

`train_word_models_soft.ipynb` chose **CNN (words)** (see `selected_model.json`) but kept its weights only in memory. This notebook rebuilds that exact model from the locked settings and saves it to disk.

**What it does**
1. Loads the same words, filters and length-balancing as `train_word_models_soft` (same seed, so the same 35,666 training words).
2. Trains the CNN for seeds 42, 43, 44 in that order, with the same architecture, optimizer, epochs and batch size.
3. **Checks the rebuild against the recorded numbers** on words-val (0.825) and HF dev (0.457). Both sets were already used for selection, so scoring them again changes nothing.
4. Saves `cnn_words_hard_s42.pt`, `cnn_words_hard_s43.pt`, `cnn_words_hard_s44.pt`.

**What it does not do:** score the HF test set. That result (PR-AUC 0.575) is already in the report and should stay a one-time score. The saved models are for Tobacco800.

**Run it in a fresh kernel** (Kernel → Restart), top to bottom, from the Run Program folder. The CNN's starting weights depend on the random state when each model is built, so nothing else should touch PyTorch's random generator first. GPU arithmetic is not perfectly repeatable, so expect small differences (about ±0.01) from the recorded numbers.

In [1]:
import json, time, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

PY_DIR = Path("PY files").resolve()
assert (PY_DIR / "model_comparison.py").exists(), f"model_comparison.py not found in {PY_DIR}"
sys.path.insert(0, str(PY_DIR))
import model_comparison as mc                     # same make_variant as the original run
from word_legibility_pipeline import clean_gray   # same cleaning as the original run
warnings.filterwarnings("ignore")

# Settings come from the locked file, not retyped
SEL = json.loads(Path("selected_model.json").read_text())
assert SEL["selected"] == "CNN (words)", SEL["selected"]
CFG = SEL["config"]
print("selected:", SEL["selected"], "| locked at", SEL["locked_at"])
print({k: CFG[k] for k in ("img_size", "epochs", "batch_size", "cnn_seeds", "dark_frac_max", "seed")})

# Numbers recorded in word_models_selection.csv, used only to check the rebuild
EXPECTED = {"val_pr_auc": 0.825, "val_roc_auc": 0.836, "dev_pr_auc": 0.457, "dev_roc_auc": 0.657}
TOL = 0.015

selected: CNN (words) | locked at 2026-11-10 17:33
{'img_size': 128, 'epochs': 10, 'batch_size': 64, 'cnn_seeds': [42, 43, 44], 'dark_frac_max': 0.35, 'seed': 42}


## 1. Training words (same filters and balancing as the original)

In [2]:
def letters_count(s):
    return pd.Series(s).astype(str).str.replace(r"[^A-Za-z]", "", regex=True).str.len().to_numpy()

W_ARR = np.load("words/words_256.npy", mmap_mode="r")
wl = pd.read_csv("words/words_labels.csv")
wl = wl[wl["similarity"].notna()].copy()
wl["n_letters"] = letters_count(wl["target"])
wl["dark_frac"] = [(np.asarray(W_ARR[int(i)]) < 128).mean() for i in wl["idx"]]
wl = wl[(wl["dark_frac"] <= CFG["dark_frac_max"]) & (wl["n_letters"] >= 4)]

tr = wl[wl.split == "train"].copy()
tr["len_bin"] = tr["n_letters"].clip(upper=10)
target = tr["label"].mean()
keep = []
for b, g in tr.groupby("len_bin"):
    pos, neg = g[g.label == 1], g[g.label == 0]
    n_pos = min(len(pos), int(round(len(neg) * target / (1 - target))))
    n_neg = min(len(neg), int(round(n_pos * (1 - target) / target)))
    keep.append(pos.sample(n_pos, random_state=CFG["seed"]))
    keep.append(neg.sample(n_neg, random_state=CFG["seed"]))
train = pd.concat(keep).sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)
val = wl[wl.split == "val"].reset_index(drop=True)

print(f"train words: {len(train)} (original run: 35666) | val words: {len(val)} (original: 4975)")
assert len(train) == 35666 and len(val) == 4975, "word set differs from the original run; check the input files"

train words: 35666 (original run: 35666) | val words: 4975 (original: 4975)


## 2. Dev signatures (for the check only)

In [3]:
splits = pd.read_csv("splits.csv")
sl = pd.read_csv("signatures_labels.csv").sort_values("idx").reset_index(drop=True)
assert len(sl) == len(splits) and (sl["idx"].to_numpy() == np.arange(len(splits))).all()
sl["split"] = splits["split"]
dev = sl[sl.split == "dev"]                      # test rows are never loaded below
S_RAW = np.load("signature_images_256.npy", mmap_mode="r")
S_DEV = np.stack([clean_gray(np.asarray(S_RAW[int(i)])) for i in dev.index])
print(f"dev signatures: {len(dev)} (legible {dev.label.mean():.3f})")

dev signatures: 1035 (legible 0.293)


## 3. Images → 128 × 128 normalized (same `make_variant` call)

In [4]:
def norm_variant(arr_like, idx):
    out = np.zeros((len(idx), CFG["img_size"], CFG["img_size"]), np.uint8)
    for k in range(0, len(idx), 2000):
        chunk = np.stack([np.asarray(arr_like[int(i)]) for i in idx[k:k + 2000]])
        out[k:k + 2000] = mc.make_variant(chunk, "norm", CFG["img_size"])
    return out

t0 = time.time()
X = {"train": norm_variant(W_ARR, train["idx"].to_numpy()),
     "val":   norm_variant(W_ARR, val["idx"].to_numpy()),
     "dev":   norm_variant(S_DEV, np.arange(len(dev)))}
Y = {"train": train.label.to_numpy(), "val": val.label.to_numpy(), "dev": dev.label.to_numpy()}
print({k: v.shape for k, v in X.items()}, f"{time.time() - t0:.0f}s")

{'train': (35666, 128, 128), 'val': (4975, 128, 128), 'dev': (1035, 128, 128)} 8s


## 4. Train the three CNNs (seeds 42, 43, 44, in that order) and save them

In [5]:
import torch
import torch.nn as nn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

def build_cnn():
    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o),
                             nn.ReLU(inplace=True), nn.MaxPool2d(2))
    return nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3),
                         nn.Linear(128, 1)).to(DEVICE)

def train_cnn(model, Xtr, ytr, epochs, seed, max_lr=3e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    pos = max(1, int(ytr.sum()))
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(len(ytr) - pos) / pos], device=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr / 3, weight_decay=1e-4)
    steps = epochs * int(np.ceil(len(ytr) / CFG["batch_size"]))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=steps)
    amp = DEVICE.type == "cuda" and not CFG["no_amp"]
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr.astype(np.float32))
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(yt)); tot = 0.0
        for s in range(0, len(perm), CFG["batch_size"]):
            idx = perm[s:s + CFG["batch_size"]]
            if len(idx) < 2: continue
            xb = (Xt[idx].to(DEVICE).float() / 255).unsqueeze(1)
            yb = yt[idx].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=amp):
                loss = loss_fn(model(xb).squeeze(1), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(idx)
        print(f"    epoch {ep + 1}/{epochs} loss {tot / len(yt):.4f}")
    return model

@torch.no_grad()
def predict_cnn(model, Xs):
    model.eval(); out = []
    amp = DEVICE.type == "cuda" and not CFG["no_amp"]
    for s in range(0, len(Xs), 256):
        xb = (torch.from_numpy(Xs[s:s + 256]).to(DEVICE).float() / 255).unsqueeze(1)
        with torch.autocast(device_type=DEVICE.type, enabled=amp):
            out.append(torch.sigmoid(model(xb).squeeze(1)).float().cpu().numpy())
    return np.concatenate(out)

SCORES, SAVED = {}, []
for sd in CFG["cnn_seeds"]:
    t = time.time(); print(f"CNN (words) seed {sd}")
    m = train_cnn(build_cnn(), X["train"], Y["train"], CFG["epochs"], sd)   # build first, as in the original
    SCORES[sd] = {k: predict_cnn(m, X[k]) for k in ("val", "dev")}
    path = f"cnn_words_hard_s{sd}.pt"
    torch.save(m.state_dict(), path); SAVED.append(path)
    print(f"  saved {path} ({time.time() - t:.0f}s)")
    del m
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

device: cuda NVIDIA GeForce RTX 5060 Laptop GPU
CNN (words) seed 42
    epoch 1/10 loss 0.5775
    epoch 2/10 loss 0.5432
    epoch 3/10 loss 0.5243
    epoch 4/10 loss 0.5134
    epoch 5/10 loss 0.4999
    epoch 6/10 loss 0.4907
    epoch 7/10 loss 0.4777
    epoch 8/10 loss 0.4675
    epoch 9/10 loss 0.4547
    epoch 10/10 loss 0.4453
  saved cnn_words_hard_s42.pt (66s)
CNN (words) seed 43
    epoch 1/10 loss 0.5760
    epoch 2/10 loss 0.5435
    epoch 3/10 loss 0.5298
    epoch 4/10 loss 0.5130
    epoch 5/10 loss 0.5008
    epoch 6/10 loss 0.4906
    epoch 7/10 loss 0.4791
    epoch 8/10 loss 0.4686
    epoch 9/10 loss 0.4565
    epoch 10/10 loss 0.4484
  saved cnn_words_hard_s43.pt (64s)
CNN (words) seed 44
    epoch 1/10 loss 0.5792
    epoch 2/10 loss 0.5428
    epoch 3/10 loss 0.5249
    epoch 4/10 loss 0.5104
    epoch 5/10 loss 0.5007
    epoch 6/10 loss 0.4916
    epoch 7/10 loss 0.4800
    epoch 8/10 loss 0.4683
    epoch 9/10 loss 0.4556
    epoch 10/10 loss 0.4492
  saved

## 5. Check the rebuild against the recorded results
Same averaging as the original: metrics per seed, then the mean over seeds.

In [6]:
rows = []
for sd, s in SCORES.items():
    rows.append({"seed": sd,
                 "val_pr_auc": average_precision_score(Y["val"], s["val"]),
                 "val_roc_auc": roc_auc_score(Y["val"], s["val"]),
                 "dev_pr_auc": average_precision_score(Y["dev"], s["dev"]),
                 "dev_roc_auc": roc_auc_score(Y["dev"], s["dev"]),
                 "dev_f1": f1_score(Y["dev"], (s["dev"] > 0.5).astype(int))})
per_seed = pd.DataFrame(rows).set_index("seed")
mean = per_seed.mean()
display(per_seed.round(3))

check = pd.DataFrame({"rebuilt": mean[list(EXPECTED)], "recorded": pd.Series(EXPECTED)})
check["diff"] = check["rebuilt"] - check["recorded"]
check["ok"] = check["diff"].abs() <= TOL
display(check.round(3))
print("REBUILD MATCHES" if check["ok"].all() else
      "Rebuild differs by more than 0.015: tell me the numbers before using these files")

Path("cnn_words_hard_manifest.json").write_text(json.dumps({
    "model": "CNN (words)", "files": SAVED, "config": CFG,
    "rebuilt_on": time.strftime("%Y-%m-%d %H:%M"),
    "check": {k: {"rebuilt": round(float(check.loc[k, "rebuilt"]), 4), "recorded": EXPECTED[k]} for k in EXPECTED},
    "per_seed": per_seed.round(4).reset_index().to_dict("records"),
    "note": "HF test not rescored; its one-time result (PR-AUC 0.575) is in word_models_holdout.csv"}, indent=1))
print("wrote cnn_words_hard_manifest.json")

,val_pr_auc,val_roc_auc,dev_pr_auc,dev_roc_auc,dev_f1
seed,,,,,
42,0.826,0.838,0.431,0.656,0.480
43,0.824,0.836,0.448,0.648,0.481
44,0.821,0.833,0.453,0.658,0.479


,rebuilt,recorded,diff,ok
val_pr_auc,0.824,0.825,-0.001,True
val_roc_auc,0.836,0.836,-0.000,True
dev_pr_auc,0.444,0.457,-0.013,True
dev_roc_auc,0.654,0.657,-0.003,True


REBUILD MATCHES
wrote cnn_words_hard_manifest.json
